In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()

# If the notebook is inside CarePulse/notebooks,
# move one level up to CarePulse
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

print(PROJECT_ROOT)

In [ ]:
INPUT_FILE = (
    PROJECT_ROOT
    / "data"
    / "cxr"
    / "cxr_labeled.csv"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "cxr"
    / "eda"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [ ]:
df = pd.read_csv(INPUT_FILE)

print("Dataset shape:", df.shape)

In [ ]:
FINDINGS = [
    "pneumonia",
    "pleural_effusion",
    "pneumothorax",
    "atelectasis",
    "consolidation",
    "pulmonary_edema",
    "cardiomegaly",
    "opacity",
]

rows = []

for finding in FINDINGS:
    col = f"label_{finding}"

    positive = (df[col] == 1).sum()
    negative = (df[col] == 0).sum()
    uncertain = (df[col] == -1).sum()
    not_mentioned = df[col].isna().sum()

    rows.append({
        "Finding": finding,
        "Positive": positive,
        "Negative": negative,
        "Uncertain": uncertain,
        "Not Mentioned": not_mentioned,
        "Usable": positive + negative,
    })

distribution_df = pd.DataFrame(rows)

distribution_df

In [ ]:
split_rows = []

for split in ["Train", "Test"]:
    split_df = df[df["split"] == split]

    for finding in FINDINGS:
        col = f"label_{finding}"

        split_rows.append({
            "Split": split,
            "Finding": finding,
            "Positive": (split_df[col] == 1).sum(),
            "Negative": (split_df[col] == 0).sum(),
            "Uncertain": (split_df[col] == -1).sum(),
        })

split_df_summary = pd.DataFrame(split_rows)

split_df_summary

In [ ]:
train_df = df[df["split"] == "Train"]

for finding in FINDINGS:

    col = f"label_{finding}"

    print("\n" + "=" * 70)
    print(f"FINDING: {finding.upper()}")
    print("=" * 70)

    print("\n--- POSITIVE ---")

    samples = train_df[train_df[col] == 1].sample(
        min(3, (train_df[col] == 1).sum()),
        random_state=42
    )

    for _, row in samples.iterrows():
        print(f"\nImage: {row['image_id']}")
        print(row["report"])

    print("\n--- NEGATIVE ---")

    samples = train_df[train_df[col] == 0].sample(
        min(3, (train_df[col] == 0).sum()),
        random_state=42
    )

    for _, row in samples.iterrows():
        print(f"\nImage: {row['image_id']}")
        print(row["report"])